This notebook is for generating "silver label" examples using the trained span identification and technique classification models in order to train a lighter weight model. The raw news article data pre-adding silver labels is from the English-only subset of the Common Crawl News dataset. Once run through the existing models to get "silver labels," we use these examples to train a xx model to be used in our Chrome extension.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datasets import load_dataset
import os
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification
import json
from tqdm.auto import tqdm
import spacy
from textblob import TextBlob
import re
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
#Identify base directory to ensure portability
BASE_DIR = Path.cwd().resolve().parent
MODELS_DIR = BASE_DIR / "models"
interim_dir = BASE_DIR / "data" / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)
output_file = interim_dir / "news_with_labels.csv"
second_output_file = Path("../data/processed/news_with_features.csv")
DATA_PATH = BASE_DIR / "data" / "processed" / "semeval_tc_cleaned.csv"

SI_DIR = MODELS_DIR / "semeval_roberta_scanner"
SI_SPEC_DIR = MODELS_DIR / "semeval_roberta_scanner_specialist"
TC_DIR = MODELS_DIR / "semeval_roberta_classifier"

SI_MODEL_PATH = f"{os.fspath(SI_DIR.absolute())}"
SI_SPEC_PATH = f"{os.fspath(SI_SPEC_DIR.absolute())}"
TC_MODEL_PATH = f"{os.fspath(TC_DIR.absolute())}"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [3]:
#Run training notebooks if models are missing
REQUIRED_FILES = ["config.json", "model.safetensors"]

def model_exists(path):
    path = Path(path)
    has_weights = any(path.glob("*.bin")) or any(path.glob("*.safetensors"))
    return has_weights

if not model_exists(SI_MODEL_PATH):
    print("SI Model missing. Running training notebook...")
    %run 4.1-fp-semeval-si-modeling.ipynb
if not model_exists(TC_MODEL_PATH):
    print("TC Model missing. Running training notebook...")
    %run 4.2-fp-semeval-tc-modeling.ipynb

In [4]:
#Load Base SI Model (RoBERTa token-classifier for span detection)
print(f"Loading Base SI Model from: {SI_MODEL_PATH}...")
si_tokenizer = AutoTokenizer.from_pretrained(SI_MODEL_PATH)
si_model = AutoModelForTokenClassification.from_pretrained(SI_MODEL_PATH, local_files_only=True).to(device)
si_model.eval()

#Load Specialist SI Model
print(f"Loading Specialist SI Model from: {SI_SPEC_PATH}...")
si_spec_model = AutoModelForTokenClassification.from_pretrained(SI_SPEC_PATH, local_files_only=True).to(device)
si_spec_model.eval()

Loading Base SI Model from: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_scanner...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading Specialist SI Model from: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_scanner_specialist...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForTokenClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (L

In [5]:
#Load TC Model (Technique Classification)
tc_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
tc_model = AutoModelForSequenceClassification.from_pretrained(TC_MODEL_PATH).to(device)
tc_model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50267, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.2, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [6]:
#Set thresholds for each technique
OPTIMIZED_THRESHOLDS = {
    'Appeal_to_Authority': 0.60,
    'Appeal_to_fear-prejudice': 0.50,
    'Bandwagon_Reductio_ad_hitlerum': 0.10,
    'Black-and-White_Fallacy': 0.20,
    'Causal_Oversimplification': 0.20,
    'Doubt': 0.35,
    'Exaggeration_Minimisation': 0.40,
    'Flag-Waving': 0.45,
    'Loaded_Language': 0.40,
    'Name_Calling_Labeling': 0.55,
    'Repetition': 0.40,
    'Slogans': 0.30,
    'Thought-terminating_Cliches': 0.15,
    'Whataboutism_Straw_Men_Red_Herring': 0.15
}

In [7]:
def run_pipeline_batched(texts):
    """
    Processes a list of texts through the SI -> Cascade -> TC pipeline.
    """
    #1. SI Tokenization (Batch)
    inputs = si_tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True
    ).to(device)

    offsets_batch = inputs.pop("offset_mapping")

    with torch.no_grad():
        #2. SI Model Inference (Batch)
        base_outputs = si_model(**inputs)
        base_preds_batch = torch.argmax(base_outputs.logits, dim=-1)

        # 3. Specialist Model Inference (Batch)
        spec_outputs = si_spec_model(**inputs)
        spec_probs_batch = F.softmax(spec_outputs.logits, dim=-1)
        propaganda_prob_batch = spec_probs_batch[:, :, 1]

    #4. Apply Cascade Logic & Extract Spans for each item in batch
    batch_final_results = []

    for i in range(len(texts)):
        text = texts[i]
        base_preds = base_preds_batch[i]
        prop_probs = propaganda_prob_batch[i]
        offsets = offsets_batch[i]

        #Merge predictions
        final_preds = base_preds.clone()
        mask = (base_preds == 0) & (prop_probs > 0.5)
        final_preds[mask] = 1

        #Extract spans
        predicted_spans = []
        current_span = None
        for j, pred in enumerate(final_preds):
            label = pred.item()
            start, end = offsets[j]
            if start == end: continue
            if label in [1, 2]:
                if current_span is None:
                    current_span = [start.item(), end.item()]
                else:
                    current_span[1] = end.item()
            elif current_span:
                predicted_spans.append(tuple(current_span))
                current_span = None
        if current_span: predicted_spans.append(tuple(current_span))

        #5. Technique Classification (TC) for extracted spans
        article_results = []
        for span in predicted_spans:
            span_text = text[span[0]:span[1]].strip()
            if not span_text: continue

            tc_inputs = tc_tokenizer(span_text, return_tensors="pt", truncation=True, padding=True).to(device)
            with torch.no_grad():
                tc_logits = tc_model(**tc_inputs).logits
                probs = torch.sigmoid(tc_logits)[0]

            found_techniques = []
            for class_id, prob in enumerate(probs):
                tech_name = tc_model.config.id2label[class_id]
                if prob.item() >= OPTIMIZED_THRESHOLDS.get(tech_name, 0.5):
                    found_techniques.append(tech_name)

            if not found_techniques:
                found_techniques.append(tc_model.config.id2label[torch.argmax(probs).item()])

            for tech in found_techniques:
                article_results.append({"span": tuple(span), "technique": tech})

        batch_final_results.append(article_results)

    return batch_final_results

In [8]:
#Load `news_with_labels.csv` if it already exists; otherwise, run the labeling pipeline and save the result
if output_file.exists():
    news = pd.read_csv(output_file, names=['text', 'propaganda'], header=0, on_bad_lines='skip')
    news["propaganda"] = news["propaganda"].apply(json.loads)
    display(news)
else:
    print("Cache not found. Downloading the data and running the RoBERTa labeling pipeline (this will take time)...")
    #Load the news article dataset
    news = load_dataset("vblagoje/cc_news", split="train")
    news = news.to_pandas()

    #Constrain to only the text, as that's the only input our extension will be given
    #And take a random sample of articles because the dataset is unreasonably large
    news = news[['text']].sample(frac=0.06, random_state=42).reset_index(drop=True)
    display(news)

    #Use run pipeline function to get predicted propaganda spans and labels from all the text
    print("Running pipeline...")
    BATCH_SIZE = 32
    all_predictions = []
    texts_to_process = news['text'].tolist()

    for i in tqdm(range(0, len(texts_to_process), BATCH_SIZE)):
        batch = texts_to_process[i : i + BATCH_SIZE]
        batch_results = run_pipeline_batched(batch)
        all_predictions.extend(batch_results)

    news['propaganda'] = all_predictions
    display(news)

    #Save the dataframe so it can be reused later
    print("Saving DataFrame...")
    news_to_save = news.copy()
    news_to_save["propaganda"] = news_to_save["propaganda"].apply(json.dumps)

    news_to_save.to_csv(output_file, index=False)
    print(f"Silver labels saved to {output_file}")

,text,propaganda
0,Nashik : Indore Infoline Pvt. Ltd has organise...,[]
1,South-East Governors on Monday re-assured Ndig...,"[{'span': [700, 707], 'technique': 'Bandwagon_..."
2,The two teenagers that were arrested in connec...,[]
3,"CHARLOTTE, North Carolina (Reuters) - Jason Da...","[{'span': [203, 211], 'technique': 'Bandwagon_..."
4,Donald Trump’s baser instincts served him well...,"[{'span': [15, 30], 'technique': 'Causal_Overs..."
...,...,...
48795,(SDOT MAP with travel times/video links; is th...,[]
48796,© Thomson Reuters 2018\nNorth Korea's growing ...,"[{'span': [901, 922], 'technique': 'Bandwagon_..."
48797,MOSCOW (AP) — Six-time Olympic gold medalist V...,"[{'span': [425, 441], 'technique': 'Flag-Wavin..."
48798,This is the miraculous moment a driver escaped...,"[{'span': [12, 22], 'technique': 'Bandwagon_Re..."


In [9]:
#Now let's add simple, cheap features to the data
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

def get_combined_features(text):
    if not isinstance(text, str) or text.strip() == "":
        return [0] * 13 #Returns 0s for all 13 features

    #1. TextBlob for Sentiment/Trustworthiness
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity

    #2. SpaCy for Linguistic & POS features
    doc = nlp(text)
    tokens = [t for t in doc if not t.is_space]
    words = [t for t in tokens if not t.is_punct]

    #Basic Counts
    char_count = len(text)
    word_count = len(words)
    total_tokens = len(tokens)
    avg_word_len = np.mean([len(w.text) for w in words]) if words else 0

    #Stylometry (Propaganda Signals)
    caps_count = sum(1 for w in words if w.text.isupper() and len(w.text) > 1)
    title_count = sum(1 for w in words if w.text.istitle())
    punct_count = sum(1 for t in tokens if any(c in '!?."' for c in t.text))
    sent_count = len(re.split(r'[.!?]+', text))

    #POS Tagging (Adjectives & Adverbs)
    adj_count = sum(1 for t in doc if t.pos_ == "ADJ")
    adv_count = sum(1 for t in doc if t.pos_ == "ADV")

    #Lexical Diversity (Type-Token Ratio)
    unique_words = len(set([w.text.lower() for w in words]))
    ttr = unique_words / max(word_count, 1)

    #3. Normalization
    caps_ratio = caps_count / max(word_count, 1)
    punct_ratio = punct_count / max(word_count, 1)
    adj_density = adj_count / max(total_tokens, 1)
    adv_density = adv_count / max(total_tokens, 1)

    return [
        char_count, word_count, avg_word_len, caps_ratio, punct_ratio,
        sent_count, title_count, total_tokens,
        polarity, subjectivity, adj_density, adv_density, ttr
    ]

In [10]:
feature_cols = ['char_count', 'word_count', 'avg_word_len', 'caps_ratio', 'punct_ratio','sent_count', 'title_count', 'total_tokens', 'polarity', 'subjectivity', 'adj_density', 'adv_density', 'lexical_diversity']

if second_output_file.exists():
    df = pd.read_csv(second_output_file)
    df.head()
else:
    print("Getting features for all articles (this will take a while)...")
    tqdm.pandas()
    features = news['text'].progress_apply(get_combined_features).tolist()

    print("Saving DataFrame...")
    features_df = pd.DataFrame(features, columns=feature_cols)
    df = pd.concat([news, features_df], axis=1)
    df.to_csv(second_output_file, index=False)
    df.head()

In [11]:
#Train/test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [12]:
#Add vectorized word counts, limit to 1000 features so the resulting CSV isn't gigabytes in size
tfidf = TfidfVectorizer(max_features=1000, stop_words='english')

#Fit on training data only to avoid data leakage
X_train_tfidf = tfidf.fit_transform(train_df['text'].fillna(""))
X_test_tfidf = tfidf.transform(test_df['text'].fillna(""))

In [13]:
#Convert TF-IDF sparse matrix to a DataFrame
tfidf_cols = [f"word_{name}" for name in tfidf.get_feature_names_out()]
train_tfidf_df = pd.DataFrame(X_train_tfidf.toarray(), columns=tfidf_cols, index=train_df.index)
test_tfidf_df = pd.DataFrame(X_test_tfidf.toarray(), columns=tfidf_cols, index=test_df.index)

In [14]:
#Converts the 'propaganda' JSON/List into a 14-column binary matrix
def get_tech_list(val):
    try:
        items = json.loads(val.replace("'", '"')) if isinstance(val, str) else val
        return [i['technique'] for i in items]
    except: return []

In [15]:
mlb = MultiLabelBinarizer()
y_train = mlb.fit_transform(train_df['propaganda'].apply(get_tech_list))
y_test = mlb.transform(test_df['propaganda'].apply(get_tech_list))

In [16]:
#Side-by-side join of feature stats and word counts
X_train = pd.concat([train_df[feature_cols], train_tfidf_df], axis=1)
X_test = pd.concat([test_df[feature_cols], test_tfidf_df], axis=1)

In [17]:
print(f"X_train shape: {X_train.shape}")
X_train.head()

X_train shape: (39040, 1013)


,char_count,word_count,avg_word_len,caps_ratio,punct_ratio,sent_count,title_count,total_tokens,polarity,subjectivity,...,word_works,word_world,word_worth,word_wrote,word_www,word_year,word_years,word_yes,word_york,word_young
40579,2225,341,5.416422,0.014663,0.029326,11,59,382,0.082374,0.322815,...,0.0,0.052025,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0
37753,2495,447,4.492170,0.000000,0.038031,18,78,497,0.039454,0.394513,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0
12791,105,18,4.833333,0.000000,0.055556,2,10,19,0.018182,0.277273,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.393492,0.0
3719,1037,166,5.162651,0.042169,0.048193,10,22,189,0.259207,0.517016,...,0.0,0.102929,0.0,0.0,0.144457,0.0,0.0,0.0,0.000000,0.0
6355,244,39,5.128205,0.025641,0.076923,4,5,46,0.300000,0.533333,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0


In [18]:
models = {
    "Logistic Regression": MultiOutputClassifier(LogisticRegression(max_iter=1000)),
    "Decision Tree": MultiOutputClassifier(DecisionTreeClassifier()),
    "Random Forest": MultiOutputClassifier(RandomForestClassifier(n_estimators=100)),
    "XGBoost": MultiOutputClassifier(XGBClassifier(use_label_encoder=False,eval_metric='logloss')) }

In [ ]:
results = {}
best_score = 0
best_model_object = None

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    #Using 'macro' F1 because we have 14 labels
    score = f1_score(y_test, preds, average='macro', zero_division=0)
    results[name] = score
    print(f"{name} F1: {score:.4f}")

    #Track the best model object
    if score > best_score:
        best_score = score
        best_model_object = model

print(f"\nThe winner is: {max(results, key=results.get)}")

In [ ]:
# Save the best model to a file
joblib.dump(best_model_object, Path("../models/lightweight_scanner/best_propaganda_model.pkl"))